<a href="https://colab.research.google.com/github/Afuhnwi-Afriitech/coin-classification/blob/main/Coin_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tqdm import tqdm


In [2]:
!pip install kaggle

from google.colab import files
import os
import json

uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename(list(uploaded.keys())[0], '/root/.kaggle/kaggle.json')

!chmod 600 /root/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [3]:
!kaggle competitions download -c dl4cv-coin-classification

!unzip dl4cv-coin-classification.zip

Streaming output truncated to the last 5000 lines.
  inflating: kaggle/train/494.jpg    
  inflating: kaggle/train/4940.jpg   
  inflating: kaggle/train/4941.jpg   
  inflating: kaggle/train/4942.jpg   
  inflating: kaggle/train/4943.jpg   
  inflating: kaggle/train/4944.jpg   
  inflating: kaggle/train/4945.jpg   
  inflating: kaggle/train/4946.jpg   
  inflating: kaggle/train/4947.jpg   
  inflating: kaggle/train/4948.jpg   
  inflating: kaggle/train/4949.jpg   
  inflating: kaggle/train/495.jpg    
  inflating: kaggle/train/4950.jpg   
  inflating: kaggle/train/4951.jpg   
  inflating: kaggle/train/4952.jpg   
  inflating: kaggle/train/4953.jpg   
  inflating: kaggle/train/4954.jpg   
  inflating: kaggle/train/4955.jpg   
  inflating: kaggle/train/4956.jpg   
  inflating: kaggle/train/4957.jpg   
  inflating: kaggle/train/4958.jpg   
  inflating: kaggle/train/4959.jpg   
  inflating: kaggle/train/496.jpg    
  inflating: kaggle/train/4960.jpg   
  inflating: kaggle/train/4961.jpg   

In [4]:
DATA_DIR = "/content/kaggle"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(train_df.head())
print("Train samples:", len(train_df))
print("Test samples:", len(test_df))


   Id                               Class
0   1  1 Cent,Australian dollar,australia
1   2  1 Cent,Australian dollar,australia
2   3  1 Cent,Australian dollar,australia
3   4  1 Cent,Australian dollar,australia
4   5  1 Cent,Australian dollar,australia
Train samples: 10368
Test samples: 1282


In [14]:
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["Class"])

NUM_CLASSES = len(label_encoder.classes_)
print("Number of classes:", NUM_CLASSES)  # should be 315


Number of classes: 315


In [ ]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)


In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])


In [ ]:
train_dir = "/content/kaggle/train"

filename_map = {}
for f in os.listdir(train_dir):
    base, ext = os.path.splitext(f)
    filename_map[base] = f


In [ ]:
from PIL import Image, UnidentifiedImageError
import os
import random

class CoinDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.is_test = is_test

        # Build filename map ONCE
        self.filename_map = {}
        for f in os.listdir(image_dir):
            base, ext = os.path.splitext(f)
            self.filename_map[base] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.loc[idx, "Id"]).strip()

        if img_id not in self.filename_map:
            # pick a random valid sample instead of crashing
            return self.__getitem__(random.randint(0, len(self.df) - 1))

        img_path = os.path.join(self.image_dir, self.filename_map[img_id])

        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            # corrupted or invalid image → resample
            return self.__getitem__(random.randint(0, len(self.df) - 1))

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_id
        else:
            label = self.df.loc[idx, "label"]
            return image, label


In [ ]:
BATCH_SIZE = 32

train_ds = CoinDataset(train_df, os.path.join(DATA_DIR, "train"), train_tfms)
val_ds   = CoinDataset(val_df,   os.path.join(DATA_DIR, "train"), val_tfms)
test_ds  = CoinDataset(test_df,  os.path.join(DATA_DIR, "test"),  val_tfms, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights="IMAGENET1K_V1")

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 125MB/s] 


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0
    correct = 0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return running_loss / len(loader), acc


def validate(model, loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()

    return correct / len(loader.dataset)


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

CKPT_DIR = "/content/drive/MyDrive/coin_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(CKPT_DIR, "checkpoint.pth")
BEST_MODEL_PATH = os.path.join(CKPT_DIR, "best_model.pth")


In [ ]:
EPOCHS = 10
best_val_acc = 0
start_epoch = 0

# 🔁 Resume if checkpoint exists
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    best_val_acc = checkpoint["best_val_acc"]
    start_epoch = checkpoint["epoch"] + 1

    print(f"Resuming from epoch {start_epoch}")

for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")

    # 🔥 Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    # 💾 Save checkpoint every epoch
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_acc": best_val_acc
    }

    torch.save(checkpoint, CHECKPOINT_PATH)


100%|██████████| 260/260 [37:39<00:00,  8.69s/it]


Epoch 1/10
Train Loss: 3.4214 | Train Acc: 0.3253
Val Acc: 0.6152


100%|██████████| 260/260 [33:28<00:00,  7.73s/it]


Epoch 2/10
Train Loss: 1.2611 | Train Acc: 0.6775
Val Acc: 0.7454


100%|██████████| 260/260 [34:13<00:00,  7.90s/it]


...

Epoch 8/10
Train Loss: 0.3194 | Train Acc: 0.9052
Val Acc: 0.8409


100%|██████████| 260/260 [33:14<00:00,  7.67s/it]


Epoch 9/10
Train Loss: 0.2658 | Train Acc: 0.9204
Val Acc: 0.8356


  7%|▋         | 17/260 [02:30<35:50,  8.85s/it]


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/coin_model_best.pth")


In [7]:
import torch
import torchvision.models as models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 315

model = models.efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

In [8]:
model.load_state_dict(torch.load("/content/drive/MyDrive/coin_model_best.pth", map_location=device))
model.eval()


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [9]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


In [20]:
from PIL import Image

img_path = "/content/kaggle/test/10075.jpg"

image = Image.open(img_path).convert("RGB")
image = transform(image)
image = image.unsqueeze(0)
image = image.to(device)

In [21]:
with torch.no_grad():
    outputs = model(image)
    pred = torch.argmax(outputs,1)

print("Predicted class:", pred.item())

Predicted class: 127


In [22]:
label = label_encoder.inverse_transform([pred.item()])
print("Predicted coin:", label[0])

Predicted coin: 2 Cents,Euro,italy
